In [11]:
from dataclasses import dataclass
from typing_extensions import Self
from typing import Callable
import uproot
import numpy as np
import awkward as ak
from tqdm import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from tqdm import tqdm
import json
import os

# %matplotlib widget

from typing import Optional

# from pileup_ml_2.pileup_ml.detectors.pixels import RowColMapping, PixelDetector, PixelModule
# from pileup_ml.pixels.hits import PixelDigiEvent
# from pileup_ml.pixels.clusters import get_module_clusters
# from pileup_ml.plots import plot_pixel_module
# from pileup_ml.visualization.pixels import visualize_event
# from pileup_ml.compression.utils import NormalizingVQWrapper

In [12]:
def read_root(path: str, branch="analyzer/digiTree") -> None:
        file = uproot.open(path)
        # ROOT file contents as awkward array
        ak_events = file[branch].arrays()
        event_lengths = ak.run_lengths(ak_events['event'])
        assert len(event_lengths) == len(np.unique(ak_events['event'])), \
            "events are not in contiguous blocks - same event appears in multiple non-adjacent groups"

        # Event row start/end indices
        event_start_idx = 0
        pixel_digis = []
        for event_length in tqdm(event_lengths):
            event_end_idx = event_start_idx + event_length

            ak_event = ak.unzip(ak_events[event_start_idx:event_end_idx])
            event_id = ak_event[0][0]  # All events in a group have the same ID
            np_event_detids = ak_event[1].to_numpy()
            np_event_rows = ak_event[2].to_numpy().astype(np.uint16)
            np_event_cols = ak_event[3].to_numpy().astype(np.uint16)
            np_event_adcs = ak_event[4].to_numpy().astype(np.uint8)

            event_start_idx = event_end_idx
            
            print(np_event_adcs)
            
            # Energy doposit event
        #     pixel_digi = PixelDigiEvent(
        #         id_=event_id,
        #         det_ids=np_event_detids,
        #         rows=np_event_rows,
        #         cols=np_event_cols,
        #         adcs=np_event_adcs,
        #         detector=detector
        #     )
        #     pixel_digis.append(pixel_digi)

        # return pixel_digis

In [13]:
path = '/data/cern/pileup_ml/premixlib2024/0001.root'
if not os.path.exists(path):
    print("Detecor premix data not found!")
    
read_root(path)

100%|██████████| 1000/1000 [00:00<00:00, 5582.73it/s]

[103  82  80 ...  71  56  42]
[62 83 93 ... 34 80 75]
[ 36 131  65 ...  33  67  57]
[ 38  55  61 ... 108  63  78]
[124 101 231 ... 255  46  73]
[127  62  61 ...  50 112  89]
[ 42  42 103 ... 144  60  39]
[ 44  73 198 ...  84  68 255]
[47 62 77 ... 34 86 86]
[47 63 96 ... 34 49 97]
[ 94 170  60 ... 117  40  37]
[ 54  83  65 ...  58 117  51]
[ 74  80  94 ... 139  74 121]
[ 42  66  54 ...  33 116  70]
[ 68  65  59 ... 255 255 255]
[ 50  77  71 ...  38 121 243]
[100  62  81 ...  54  41  80]
[ 40 158  70 ... 174 179 114]
[ 49  93  76 ...  41 133  77]
[142  72  92 ...  96  40  37]
[ 51  98  38 ...  38  33 115]
[ 54  60  44 ...  43 142  62]
[ 47  49  97 ...  30 159  46]
[71 96 44 ... 67 55 46]
[116 255 240 ...  58  44  95]
[ 49 139 105 ...  76  52  64]
[57 65 45 ... 30 90 94]
[ 57  44  39 ...  31  96 191]
[138 163  93 ...  33  33  55]
[ 35  46  35 ...  47  32 136]
[ 53 105  92 ...  62 196  36]
[137  58  82 ...  98 124 255]
[60 64 72 ... 82 62 50]
[87 54 76 ... 55 33 36]
[ 50  73  60 ... 124  

In [ ]:
# TODO
# Try reading hit data. Create a matrix form which could be suitable for 
# VQ-VAE compression
# COmpress adcs values like with Kmeans. Find appropriate cendroids for adcs
# Do not use coordinates. They may distort spatial values
